# Movie Recommendation System

Builds a content-based recommender using **TF-IDF** on movie metadata (genres, keywords, tagline, cast, director) and **cosine similarity** to rank matches.

## Step 1 — Load libraries

In [ ]:
import numpy as np
import pandas as pd
import difflib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## Step 2 — Load the dataset

In [ ]:
df = pd.read_csv('/content/movies.csv')
print(df.shape)
df.head()


## Step 3 — Select metadata columns for similarity

In [ ]:
cols = ['genres', 'keywords', 'tagline', 'cast', 'director']

# fill missing values so string concat doesn't break
for col in cols:
    df[col] = df[col].fillna('')


## Step 4 — Merge all metadata into one text field per movie

In [ ]:
df['text'] = (df['genres'] + ' ' + df['keywords'] + ' '
              + df['tagline'] + ' ' + df['cast'] + ' ' + df['director'])

df['text'].head(3)


## Step 5 — TF-IDF vectorisation

Converts each movie's text blob into a weighted term vector. Rare, distinctive words get higher weight than common ones.

In [ ]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['text'])

print('Matrix shape:', tfidf_matrix.shape)


## Step 6 — Cosine similarity matrix

In [ ]:
sim = cosine_similarity(tfidf_matrix)
print('Similarity matrix shape:', sim.shape)


## Step 7 — Take user input and find the closest title

In [ ]:
movie_name = input('Enter a movie name: ')

all_titles = df['title'].tolist()
matches = difflib.get_close_matches(movie_name, all_titles)

if not matches:
    print('No close match found. Try another title.')
else:
    best = matches[0]
    print(f'Matched to: {best}')


## Step 8 — Retrieve and rank similar movies

In [ ]:
idx = df[df['title'] == best].index[0]

scores = list(enumerate(sim[idx]))
ranked = sorted(scores, key=lambda x: x[1], reverse=True)

print('Top recommendations:\n')
count = 1
for movie in ranked:
    title = df.iloc[movie[0]]['title']
    if title != best and count <= 15:
        print(f'{count}. {title}')
        count += 1


## Full pipeline — run from scratch

In [ ]:
movie_name = input('Enter a movie name: ')

all_titles = df['title'].tolist()
matches = difflib.get_close_matches(movie_name, all_titles)

if not matches:
    print('No match found.')
else:
    best = matches[0]
    idx = df[df['title'] == best].index[0]
    scores = list(enumerate(sim[idx]))
    ranked = sorted(scores, key=lambda x: x[1], reverse=True)

    print(f'Because you liked "{best}", you might enjoy:\n')
    count = 1
    for movie in ranked:
        title = df.iloc[movie[0]]['title']
        if title != best and count <= 15:
            print(f'{count}. {title}')
            count += 1
